# 02 - Anomaly Dataset Creation

**Thesis:** Structured Subsystem-Aware Feature Representations for Predictive Maintenance in Metro Systems
**Author:** Emirhan Kurtulus, TU Wien

This notebook creates the anomaly dataset by:
1. Creating supervised windowed features (Steiner's best config: Wfeat=350s, Wlabel=22326s)
2. Training a Random Forest classifier on the train set
3. Predicting on ALL test windows (no filtering)
4. Creating unsupervised sequences (Wseq=1402s, z-score normalized)
5. Training an LSTM-AE on normal-only train sequences
6. Computing reconstruction-error anomaly scores on ALL test sequences
7. Saving the full anomaly dataset for downstream analysis

**Output:** A CSV with all test windows containing features, true labels, RF predictions, and LSTM-AE anomaly scores.

In [ ]:
import sys, os, gc, bisect, itertools, json, warnings
import numpy as np
import pandas as pd
import glob
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
import joblib

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

ROOT = os.path.abspath('..')
TRAIN_GLOB = os.path.join(ROOT, 'train', '**', '*.parquet')
TEST_GLOB  = os.path.join(ROOT, 'test',  '**', '*.parquet')
OUT_DIR    = os.path.join(ROOT, 'outputs')
os.makedirs(OUT_DIR, exist_ok=True)

RNG = np.random.RandomState(42)

# ── Steiner's best config ──
WINDOW_S   = 350      # 6 min feature aggregation window
HORIZON_S  = 22326    # 6.2 h failure label shift
TS_COL     = 'TIMESTAMP'
FAIL_COL   = 'TRAIN_IS_IN_FAILURE'
MAINT_COL  = 'TRAIN_IS_IN_MAINTENANCE'
FAIL_TYPE  = 'TRAIN_FAILURE_TYPE'

# ── Sensor classification (matching Steiner's 95 sensors) ──
# 74 analog sensors: continuous values -> min/max/mean/sum
ANALOG_SENSORS = [
    # APU (2)
    'CW1_MAIN_RESERVOIR_PRESSURE', 'CW2_MAIN_RESERVOIR_PRESSURE',
    # Brake cylinder pressure (12)
    'CW1_BRAKE_CYLINDER_PRESSURE_BOGIE1', 'CW1_BRAKE_CYLINDER_PRESSURE_BOGIE2',
    'CW2_BRAKE_CYLINDER_PRESSURE_BOGIE1', 'CW2_BRAKE_CYLINDER_PRESSURE_BOGIE2',
    'MW1_BRAKE_CYLINDER_PRESSURE_BOGIE1', 'MW1_BRAKE_CYLINDER_PRESSURE_BOGIE2',
    'MW2_BRAKE_CYLINDER_PRESSURE_BOGIE1', 'MW2_BRAKE_CYLINDER_PRESSURE_BOGIE2',
    'MW3_BRAKE_CYLINDER_PRESSURE_BOGIE1', 'MW3_BRAKE_CYLINDER_PRESSURE_BOGIE2',
    'MW4_BRAKE_CYLINDER_PRESSURE_BOGIE1', 'MW4_BRAKE_CYLINDER_PRESSURE_BOGIE2',
    # Spring brake pressure (12)
    'CW1_SPRING_BRAKE_PRESSURE_BOGIE1', 'CW1_SPRING_BRAKE_PRESSURE_BOGIE2',
    'CW2_SPRING_BRAKE_PRESSURE_BOGIE1', 'CW2_SPRING_BRAKE_PRESSURE_BOGIE2',
    'MW1_SPRING_BRAKE_PRESSURE_BOGIE1', 'MW1_SPRING_BRAKE_PRESSURE_BOGIE2',
    'MW2_SPRING_BRAKE_PRESSURE_BOGIE1', 'MW2_SPRING_BRAKE_PRESSURE_BOGIE2',
    'MW3_SPRING_BRAKE_PRESSURE_BOGIE1', 'MW3_SPRING_BRAKE_PRESSURE_BOGIE2',
    'MW4_SPRING_BRAKE_PRESSURE_BOGIE1', 'MW4_SPRING_BRAKE_PRESSURE_BOGIE2',
    # Proportional valve pressure (12)
    'CW1_PROPORTIONAL_VALVE_PRESSURE_BOGIE1', 'CW1_PROPORTIONAL_VALVE_PRESSURE_BOGIE2',
    'CW2_PROPORTIONAL_VALVE_PRESSURE_BOGIE1', 'CW2_PROPORTIONAL_VALVE_PRESSURE_BOGIE2',
    'MW1_PROPORTIONAL_VALVE_PRESSURE_BOGIE1', 'MW1_PROPORTIONAL_VALVE_PRESSURE_BOGIE2',
    'MW2_PROPORTIONAL_VALVE_PRESSURE_BOGIE1', 'MW2_PROPORTIONAL_VALVE_PRESSURE_BOGIE2',
    'MW3_PROPORTIONAL_VALVE_PRESSURE_BOGIE1', 'MW3_PROPORTIONAL_VALVE_PRESSURE_BOGIE2',
    'MW4_PROPORTIONAL_VALVE_PRESSURE_BOGIE1', 'MW4_PROPORTIONAL_VALVE_PRESSURE_BOGIE2',
    # Pneumatic braking force (12)
    'CW1_PNEUMATIC_BRAKING_FORCE_BOGIE1', 'CW1_PNEUMATIC_BRAKING_FORCE_BOGIE2',
    'CW2_PNEUMATIC_BRAKING_FORCE_BOGIE1', 'CW2_PNEUMATIC_BRAKING_FORCE_BOGIE2',
    'MW1_PNEUMATIC_BRAKING_FORCE_BOGIE1', 'MW1_PNEUMATIC_BRAKING_FORCE_BOGIE2',
    'MW2_PNEUMATIC_BRAKING_FORCE_BOGIE1', 'MW2_PNEUMATIC_BRAKING_FORCE_BOGIE2',
    'MW3_PNEUMATIC_BRAKING_FORCE_BOGIE1', 'MW3_PNEUMATIC_BRAKING_FORCE_BOGIE2',
    'MW4_PNEUMATIC_BRAKING_FORCE_BOGIE1', 'MW4_PNEUMATIC_BRAKING_FORCE_BOGIE2',
    # Load pressure (12)
    'CW1_LOAD_PRESSURE_BOGIE1', 'CW1_LOAD_PRESSURE_BOGIE2',
    'CW2_LOAD_PRESSURE_BOGIE1', 'CW2_LOAD_PRESSURE_BOGIE2',
    'MW1_LOAD_PRESSURE_BOGIE1', 'MW1_LOAD_PRESSURE_BOGIE2',
    'MW2_LOAD_PRESSURE_BOGIE1', 'MW2_LOAD_PRESSURE_BOGIE2',
    'MW3_LOAD_PRESSURE_BOGIE1', 'MW3_LOAD_PRESSURE_BOGIE2',
    'MW4_LOAD_PRESSURE_BOGIE1', 'MW4_LOAD_PRESSURE_BOGIE2',
    # Load signal (6)
    'CW1_LOAD_SIGNAL', 'CW2_LOAD_SIGNAL',
    'MW1_LOAD_SIGNAL', 'MW2_LOAD_SIGNAL', 'MW3_LOAD_SIGNAL', 'MW4_LOAD_SIGNAL',
    # Traction (4)
    'MW1_ENERGY_BRAKING_RESISTANCE', 'MW2_ENERGY_BRAKING_RESISTANCE',
    'MW3_ENERGY_BRAKING_RESISTANCE', 'MW4_ENERGY_BRAKING_RESISTANCE',
    # Context (2)
    'TRAIN_SPEED_ACTUAL', 'AMBIENT_TEMPERATURE',
]

# 21 binary sensors: 0/1 flags -> active_s/flips
BINARY_SENSORS = [
    # Pneumatic brake active (2)
    'CW1_PNEUMATIC_BRAKE_ACTIVE', 'CW2_PNEUMATIC_BRAKE_ACTIVE',
    # Spring brake active (4)
    'CW1_SPRING_BRAKE_ACTIVE_BOGIE1', 'CW1_SPRING_BRAKE_ACTIVE_BOGIE2',
    'CW2_SPRING_BRAKE_ACTIVE_BOGIE1', 'CW2_SPRING_BRAKE_ACTIVE_BOGIE2',
    # Proportional valve pressure available (12)
    'CW1_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE1', 'CW1_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE2',
    'CW2_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE1', 'CW2_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE2',
    'MW1_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE1', 'MW1_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE2',
    'MW2_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE1', 'MW2_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE2',
    'MW3_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE1', 'MW3_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE2',
    'MW4_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE1', 'MW4_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE2',
    # Compressor running (2)
    'CW1_COMPRESSOR_RUNNING', 'CW2_COMPRESSOR_RUNNING',
    # Brake signal (1)
    'TRAIN_BRAKE_SIGNAL',
]

ALL_SENSORS = ANALOG_SENSORS + BINARY_SENSORS

print(f'Analog sensors: {len(ANALOG_SENSORS)}')
print(f'Binary sensors: {len(BINARY_SENSORS)}')
print(f'Total sensors:  {len(ALL_SENSORS)}')
print(f'Expected features: {len(ANALOG_SENSORS)*4} analog + {len(BINARY_SENSORS)*2} binary + 2 asset = {len(ANALOG_SENSORS)*4 + len(BINARY_SENSORS)*2 + 2}')
print(f'Window: {WINDOW_S}s ({WINDOW_S/60:.1f} min) | Horizon: {HORIZON_S}s ({HORIZON_S/3600:.1f} h)')

## 1. Supervised Feature Dataset Creation

Replicates Steiner's pipeline:
- Epoch-aligned non-overlapping windows (350s)
- Filter out maintenance periods
- Analog: min/max/mean/sum per window
- Binary: active_s (seconds active) / flips (state transitions) per window
- Asset lookback: days_since_last_failure, days_since_last_revision
- Shifted failure labels (6.2h horizon) + filter rule

In [ ]:
# ── Helper functions ──

def get_failure_events(df):
    """Extract continuous failure event blocks."""
    mask = df[FAIL_COL].astype(bool)
    grp = (mask != mask.shift()).cumsum()
    events = (
        df[mask].groupby(grp[mask])
        .agg(start=(TS_COL, 'min'), end=(TS_COL, 'max'),
             event_type=(FAIL_TYPE, 'first'), n_rows=(TS_COL, 'count'))
        .reset_index(drop=True)
    )
    return events


def get_maintenance_events(df):
    """Extract continuous maintenance event blocks."""
    mask = df[MAINT_COL].astype(bool)
    grp = (mask != mask.shift()).cumsum()
    events = (
        df[mask].groupby(grp[mask])
        .agg(start=(TS_COL, 'min'), end=(TS_COL, 'max'), n_rows=(TS_COL, 'count'))
        .reset_index(drop=True)
    )
    return events


def create_windowed_features(df, window_s, analog_sensors, binary_sensors):
    """Create epoch-aligned non-overlapping windowed features using vectorized ops."""
    df = df.set_index(TS_COL).sort_index()
    
    # Pre-compute flip indicators for binary sensors
    flip_cols = []
    for col in binary_sensors:
        fcol = f'_flip_{col}'
        df[fcol] = df[col].diff().abs().fillna(0)
        flip_cols.append(fcol)
    
    # Epoch-aligned resampling
    resampler = df.resample(f'{window_s}s', origin='epoch')
    
    # Analog aggregations
    print('  Computing analog features...')
    analog_min  = resampler[analog_sensors].min()
    analog_max  = resampler[analog_sensors].max()
    analog_mean = resampler[analog_sensors].mean()
    analog_sum  = resampler[analog_sensors].sum()
    
    analog_min.columns  = [f'{c}_min' for c in analog_sensors]
    analog_max.columns  = [f'{c}_max' for c in analog_sensors]
    analog_mean.columns = [f'{c}_mean' for c in analog_sensors]
    analog_sum.columns  = [f'{c}_sum' for c in analog_sensors]
    
    # Binary aggregations
    print('  Computing binary features...')
    binary_active = resampler[binary_sensors].sum()
    binary_flips  = resampler[flip_cols].sum()
    
    binary_active.columns = [f'{c}_active_s' for c in binary_sensors]
    binary_flips.columns  = [f'{c}_flips' for c in binary_sensors]
    
    # Count rows per window (to filter empty windows)
    counts = resampler[analog_sensors[0]].count().rename('_n_rows')
    
    # Combine
    feats = pd.concat([analog_min, analog_max, analog_mean, analog_sum,
                       binary_active, binary_flips, counts], axis=1)
    
    # Remove empty windows
    feats = feats[feats['_n_rows'] > 0].drop(columns=['_n_rows'])
    
    # Add timestamp and window_end columns
    feats['window_end'] = feats.index + pd.Timedelta(seconds=window_s)
    feats = feats.reset_index().rename(columns={TS_COL: 'timestamp'})
    
    print(f'  Created {len(feats)} windows with {len(feats.columns)-2} features')
    return feats


def compute_shifted_labels(window_ends, failure_events, horizon_s):
    """For each window_end, label=1 if any failure overlaps [window_end, window_end + horizon)."""
    we_arr = pd.to_datetime(window_ends).values  # numpy datetime64 array
    labels = np.zeros(len(we_arr), dtype=np.int8)
    horizon_td = np.timedelta64(horizon_s, 's')
    
    for _, fe in failure_events.iterrows():
        fs = np.datetime64(fe['start'])
        fe_end = np.datetime64(fe['end'])
        mask = (fs < we_arr + horizon_td) & (fe_end >= we_arr)
        labels[mask] = 1
    
    return labels


def compute_current_failure(window_ends, failure_events):
    """For each window_end, in_failure=1 if it falls within any failure interval."""
    we_arr = pd.to_datetime(window_ends).values
    labels = np.zeros(len(we_arr), dtype=np.int8)
    
    for _, fe in failure_events.iterrows():
        fs = np.datetime64(fe['start'])
        fe_end = np.datetime64(fe['end'])
        mask = (we_arr >= fs) & (we_arr <= fe_end)
        labels[mask] = 1
    
    return labels


def compute_asset_lookback(window_ends, fail_events, maint_events):
    """Compute days_since_last_failure and days_since_last_revision."""
    fail_ends = sorted(fail_events['end'].values)
    maint_ends = sorted(maint_events['end'].values)
    we_arr = pd.to_datetime(window_ends).values
    
    dsf = np.full(len(we_arr), np.nan, dtype=np.float64)
    dsr = np.full(len(we_arr), np.nan, dtype=np.float64)
    
    for i, we in enumerate(we_arr):
        idx = bisect.bisect_right(fail_ends, we) - 1
        if idx >= 0:
            dsf[i] = (we - fail_ends[idx]) / np.timedelta64(1, 'D')
        
        idx = bisect.bisect_right(maint_ends, we) - 1
        if idx >= 0:
            dsr[i] = (we - maint_ends[idx]) / np.timedelta64(1, 'D')
    
    return dsf, dsr


print('Helper functions defined.')

In [ ]:
# ── Load TRAIN data and create features ──
print('Loading train data...')
train_files = sorted(glob.glob(TRAIN_GLOB, recursive=True))
keep_cols = [TS_COL, FAIL_COL, FAIL_TYPE, MAINT_COL] + ALL_SENSORS

chunks = []
for i, f in enumerate(train_files):
    df = pd.read_parquet(f)
    avail_cols = [c for c in keep_cols if c in df.columns]
    chunks.append(df[avail_cols])
    del df
    if (i + 1) % 50 == 0:
        print(f'  Loaded {i+1}/{len(train_files)} files')

train_raw = pd.concat(chunks, ignore_index=True)
del chunks; gc.collect()

train_raw[TS_COL] = pd.to_datetime(train_raw[TS_COL])
train_raw.sort_values(TS_COL, inplace=True)
train_raw.reset_index(drop=True, inplace=True)
print(f'Train: {len(train_raw):,} rows | {train_raw[TS_COL].min()} to {train_raw[TS_COL].max()}')

# Extract events before filtering
train_fail_events  = get_failure_events(train_raw)
train_maint_events = get_maintenance_events(train_raw)
print(f'Train failure events: {len(train_fail_events)} | Maintenance events: {len(train_maint_events)}')

# Filter out maintenance periods
train_clean = train_raw[~train_raw[MAINT_COL].astype(bool)].copy()
print(f'After maintenance filter: {len(train_clean):,} rows')
del train_raw; gc.collect()

# Create windowed features
print('Creating windowed features...')
feats_train = create_windowed_features(train_clean, WINDOW_S, ANALOG_SENSORS, BINARY_SENSORS)
del train_clean; gc.collect()

# Add asset lookback
print('Computing asset lookback...')
dsf, dsr = compute_asset_lookback(feats_train['window_end'], train_fail_events, train_maint_events)
feats_train['days_since_last_failure'] = dsf
feats_train['days_since_last_revision'] = dsr

# Compute labels
print('Computing shifted labels...')
feats_train['label'] = compute_shifted_labels(feats_train['window_end'], train_fail_events, HORIZON_S)
feats_train['_in_failure'] = compute_current_failure(feats_train['window_end'], train_fail_events)

# Apply filter rule: keep label=1 OR (label=0 AND not currently in failure)
mask = (feats_train['label'] == 1) | ((feats_train['label'] == 0) & (feats_train['_in_failure'] == 0))
sup_train = feats_train[mask].drop(columns=['_in_failure']).reset_index(drop=True)

print(f'\nSupervised train set: {len(sup_train)} windows')
print(f'  Failure (label=1): {(sup_train["label"]==1).sum()} ({100*(sup_train["label"]==1).mean():.2f}%)')
print(f'  Normal  (label=0): {(sup_train["label"]==0).sum()} ({100*(sup_train["label"]==0).mean():.2f}%)')
print(f'  Features: {len(sup_train.columns) - 3} (excl. timestamp, window_end, label)')

# Save
os.makedirs(os.path.join(OUT_DIR, 'supervised'), exist_ok=True)
sup_train.to_parquet(os.path.join(OUT_DIR, 'supervised', 'train.parquet'), index=False)
print(f'Saved to outputs/supervised/train.parquet')

del feats_train; gc.collect()

In [ ]:
# ── Load TEST data and create features ──
print('Loading test data...')
test_files = sorted(glob.glob(TEST_GLOB, recursive=True))

chunks = []
for f in test_files:
    df = pd.read_parquet(f)
    avail_cols = [c for c in keep_cols if c in df.columns]
    chunks.append(df[avail_cols])
    del df

test_raw = pd.concat(chunks, ignore_index=True)
del chunks; gc.collect()

test_raw[TS_COL] = pd.to_datetime(test_raw[TS_COL])
test_raw.sort_values(TS_COL, inplace=True)
test_raw.reset_index(drop=True, inplace=True)
print(f'Test: {len(test_raw):,} rows | {test_raw[TS_COL].min()} to {test_raw[TS_COL].max()}')

# Extract events
test_fail_events  = get_failure_events(test_raw)
test_maint_events = get_maintenance_events(test_raw)
print(f'Test failure events: {len(test_fail_events)} | Maintenance events: {len(test_maint_events)}')

# Combine failure/maintenance events from train + test for asset lookback
all_fail_events  = pd.concat([train_fail_events, test_fail_events], ignore_index=True)
all_maint_events = pd.concat([train_maint_events, test_maint_events], ignore_index=True)

# Filter out maintenance periods
test_clean = test_raw[~test_raw[MAINT_COL].astype(bool)].copy()
print(f'After maintenance filter: {len(test_clean):,} rows')
del test_raw; gc.collect()

# Create windowed features
print('Creating windowed features...')
feats_test = create_windowed_features(test_clean, WINDOW_S, ANALOG_SENSORS, BINARY_SENSORS)
del test_clean; gc.collect()

# Add asset lookback (using combined train+test events for continuity)
print('Computing asset lookback...')
dsf, dsr = compute_asset_lookback(feats_test['window_end'], all_fail_events, all_maint_events)
feats_test['days_since_last_failure'] = dsf
feats_test['days_since_last_revision'] = dsr

# Compute labels
print('Computing shifted labels...')
feats_test['label'] = compute_shifted_labels(feats_test['window_end'], test_fail_events, HORIZON_S)
feats_test['_in_failure'] = compute_current_failure(feats_test['window_end'], test_fail_events)

# Apply filter rule
mask = (feats_test['label'] == 1) | ((feats_test['label'] == 0) & (feats_test['_in_failure'] == 0))
sup_test = feats_test[mask].drop(columns=['_in_failure']).reset_index(drop=True)

print(f'\nSupervised test set: {len(sup_test)} windows')
print(f'  Failure (label=1): {(sup_test["label"]==1).sum()} ({100*(sup_test["label"]==1).mean():.2f}%)')
print(f'  Normal  (label=0): {(sup_test["label"]==0).sum()} ({100*(sup_test["label"]==0).mean():.2f}%)')

# Save
sup_test.to_parquet(os.path.join(OUT_DIR, 'supervised', 'test.parquet'), index=False)
print(f'Saved to outputs/supervised/test.parquet')

# Also save full test (without filter rule) for later analysis
feats_test_full = feats_test.drop(columns=['_in_failure']).reset_index(drop=True)
feats_test_full.to_parquet(os.path.join(OUT_DIR, 'supervised', 'test_full_unfiltered.parquet'), index=False)
print(f'Saved unfiltered test ({len(feats_test_full)} windows) to outputs/supervised/test_full_unfiltered.parquet')

del feats_test; gc.collect()

In [ ]:
# ── Verify supervised datasets ──
sup_train = pd.read_parquet(os.path.join(OUT_DIR, 'supervised', 'train.parquet'))
sup_test  = pd.read_parquet(os.path.join(OUT_DIR, 'supervised', 'test.parquet'))

print('=== Supervised Dataset Summary ===')
print(f'Train: {sup_train.shape} | Test: {sup_test.shape}')
print(f'\nTrain labels: {sup_train["label"].value_counts().to_dict()}')
print(f'Test labels:  {sup_test["label"].value_counts().to_dict()}')
print(f'\nTrain date range: {sup_train["timestamp"].min()} to {sup_train["timestamp"].max()}')
print(f'Test date range:  {sup_test["timestamp"].min()} to {sup_test["timestamp"].max()}')
print(f'\nFeature columns: {len(sup_train.columns) - 3}')
print(f'Columns: {sup_train.columns.tolist()[:10]}...')
print(f'\nMissing values in train: {sup_train.isnull().sum().sum()}')
print(f'Missing values in test:  {sup_test.isnull().sum().sum()}')

## 2. Random Forest Training

Replicates Steiner's RF pipeline:
- Undersample majority class to 10% positive ratio
- 5-fold blocked (temporal) CV with grid search
- Threshold tuning on dev holdout (last 20%)
- Refit on full train, predict on ALL test windows

In [ ]:
# ── RF Training (using Steiner's best hyperparameters) ──
sup_train = pd.read_parquet(os.path.join(OUT_DIR, 'supervised', 'train.parquet'))
sup_train['timestamp'] = pd.to_datetime(sup_train['timestamp'])
sup_train['window_end'] = pd.to_datetime(sup_train['window_end'])
sup_train = sup_train.sort_values('window_end').reset_index(drop=True)

LABEL_COL = 'label'
TIME_COL  = 'window_end'
ID_COLS   = {'timestamp', 'window_end'}
MIN_POS_RATIO = 0.10

# Steiner's best params for 6m_22326_shift config
BEST_PARAMS = {
    'n_estimators': 300,
    'max_depth': 4,
    'min_samples_split': 10,
    'min_samples_leaf': 4,
    'max_features': 'sqrt',
    'class_weight': None,
}

feature_cols = [c for c in sup_train.columns if c not in ID_COLS | {LABEL_COL}]
print(f'Feature columns: {len(feature_cols)}')
print(f'Using Steiner best params: {BEST_PARAMS}')


def undersample(df, min_pos_ratio, rng):
    pos_idx = np.flatnonzero(df[LABEL_COL] == 1)
    neg_idx = np.flatnonzero(df[LABEL_COL] == 0)
    n_pos = len(pos_idx)
    if n_pos == 0 or n_pos / (n_pos + len(neg_idx)) >= min_pos_ratio:
        return df
    max_neg = int(np.floor(n_pos * (1.0 / min_pos_ratio - 1.0)))
    keep_neg = rng.choice(neg_idx, size=min(len(neg_idx), max_neg), replace=False)
    keep = np.sort(np.concatenate([pos_idx, keep_neg]))
    return df.iloc[keep]


def eval_best_threshold(y_true, y_proba):
    best_f1, best_thr, best_p, best_r = -1, 0.5, 0, 0
    for thr in np.linspace(0.01, 0.99, 99):
        yp = (y_proba >= thr).astype(int)
        f1 = f1_score(y_true, yp, zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = f1, float(thr)
            best_p = precision_score(y_true, yp, zero_division=0)
            best_r = recall_score(y_true, yp, zero_division=0)
    return best_thr, best_f1, best_p, best_r


# Undersample
pdf = undersample(sup_train, MIN_POS_RATIO, RNG)
print(f'After undersampling: {len(pdf)} windows (pos ratio: {(pdf[LABEL_COL]==1).mean():.3f})')

# Dev/holdout split (80/20 temporal) for threshold tuning
holdout_size = int(0.2 * len(pdf))
pdf_dev = pdf.iloc[:-holdout_size].reset_index(drop=True)
pdf_holdout = pdf.iloc[-holdout_size:].reset_index(drop=True)
print(f'Dev: {len(pdf_dev)} | Holdout: {len(pdf_holdout)}')

# Build preprocessor
pre = ColumnTransformer(
    transformers=[('pre', Pipeline([('impute', SimpleImputer(strategy='median', keep_empty_features=True))]), feature_cols)],
    verbose_feature_names_out=False
)

X_dev = pdf_dev[feature_cols]
y_dev = pdf_dev[LABEL_COL].astype(int)
X_holdout = pdf_holdout[feature_cols]
y_holdout = pdf_holdout[LABEL_COL].astype(int)

# Train on dev set with Steiner's best params
print('\nTraining RF with Steiner best params...')
final_model = Pipeline([
    ('pre', pre),
    ('clf', RandomForestClassifier(random_state=42, n_jobs=-1, **BEST_PARAMS))
])
final_model.fit(X_dev, y_dev)

# Tune threshold on holdout
proba_holdout = final_model.predict_proba(X_holdout)[:, 1]
best_thr, f1_h, p_h, r_h = eval_best_threshold(y_holdout.values, proba_holdout)
print(f'Dev holdout: F1={f1_h:.3f} P={p_h:.3f} R={r_h:.3f} thr={best_thr:.3f}')

# Refit on full undersampled train
print('Refitting on full undersampled train...')
X_full = pdf[feature_cols]
y_full = pdf[LABEL_COL].astype(int)
final_model.fit(X_full, y_full)

# Save model
model_path = os.path.join(OUT_DIR, 'supervised', 'rf_model.joblib')
joblib.dump({'model': final_model, 'threshold': best_thr, 'params': BEST_PARAMS,
             'dev_metrics': {'F1': f1_h, 'P': p_h, 'R': r_h, 'thr': best_thr}},
            model_path)
print(f'Model saved to {model_path}')

del pdf, pdf_dev, pdf_holdout, X_dev, y_dev, X_holdout, y_holdout, X_full, y_full
gc.collect()

In [ ]:
# ── RF Predict on ALL test windows ──
saved = joblib.load(os.path.join(OUT_DIR, 'supervised', 'rf_model.joblib'))
final_model = saved['model']
best_thr = saved['threshold']

sup_test = pd.read_parquet(os.path.join(OUT_DIR, 'supervised', 'test.parquet'))
sup_test['timestamp'] = pd.to_datetime(sup_test['timestamp'])
sup_test['window_end'] = pd.to_datetime(sup_test['window_end'])

feature_cols = [c for c in sup_test.columns if c not in {'timestamp', 'window_end', 'label'}]

X_test = sup_test[feature_cols]
y_test = sup_test['label'].astype(int)

proba_test = final_model.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= best_thr).astype(int)

# Metrics
f1_t = f1_score(y_test, pred_test, zero_division=0)
p_t = precision_score(y_test, pred_test, zero_division=0)
r_t = recall_score(y_test, pred_test, zero_division=0)
cm = confusion_matrix(y_test, pred_test)
tn, fp, fn, tp = cm.ravel()
far = fp / (fp + tn) if (fp + tn) > 0 else 0

print(f'=== RF Test Results (threshold={best_thr:.3f}) ===')
print(f'  F1={f1_t:.3f}  P={p_t:.3f}  R={r_t:.3f}  FAR={far:.4f}')
print(f'  TP={tp}  FP={fp}  TN={tn}  FN={fn}')
print(f'  True failures: {y_test.sum()} | Predicted: {pred_test.sum()}')

# Build output CSV
out_df = sup_test.copy()
out_df['true_failure_label'] = y_test.values
out_df['predicted_failure_label'] = pred_test
out_df['probability_of_failure'] = proba_test
out_df.drop(columns=['label'], inplace=True)

out_df.to_csv(os.path.join(OUT_DIR, 'supervised', 'test_predictions_rf.csv'), index=False)
print(f'\nSaved RF predictions: {len(out_df)} windows to outputs/supervised/test_predictions_rf.csv')

del X_test; gc.collect()

In [ ]:
# ── Create unsupervised sequences ──
SEQ_WINDOW_S = 1402  # 23 min
UNSUP_CHANNELS = ALL_SENSORS  # same 95 sensors
SUBSAMPLE = 4  # take every 4th timestep: 1402 -> 350 (saves 4x memory)

print('Loading train data for unsupervised sequences...')
train_files = sorted(glob.glob(TRAIN_GLOB, recursive=True))
unsup_cols = [TS_COL, FAIL_COL, MAINT_COL] + UNSUP_CHANNELS

chunks = []
for f in train_files:
    df = pd.read_parquet(f)
    avail = [c for c in unsup_cols if c in df.columns]
    chunks.append(df[avail])
    del df

train_raw = pd.concat(chunks, ignore_index=True)
del chunks; gc.collect()

train_raw[TS_COL] = pd.to_datetime(train_raw[TS_COL])
train_raw.sort_values(TS_COL, inplace=True)
train_raw.reset_index(drop=True, inplace=True)

# Filter maintenance
train_clean = train_raw[~train_raw[MAINT_COL].astype(bool)].copy()
del train_raw; gc.collect()

# Identify normal rows (not in failure)
train_normal = train_clean[~train_clean[FAIL_COL].astype(bool)]

# Compute z-score stats from normal train data
print('Computing z-score statistics from normal train data...')
channel_stats = {}
for c in UNSUP_CHANNELS:
    mu = float(train_normal[c].mean())
    sd = float(train_normal[c].std())
    if sd == 0 or np.isnan(sd):
        sd = 1.0
    channel_stats[c] = {'mean': mu, 'std': sd}

del train_normal; gc.collect()

# z-score standardize
print('Standardizing train data...')
for c in UNSUP_CHANNELS:
    train_clean[c] = (train_clean[c] - channel_stats[c]['mean']) / channel_stats[c]['std']

# Create non-overlapping sequences with subsampling to reduce memory
# Output shape per sequence: (1402//4, 95) = (350, 95) instead of (1402, 95)
print(f'Creating train sequences (subsample={SUBSAMPLE}, effective length={SEQ_WINDOW_S // SUBSAMPLE})...')
train_clean = train_clean.set_index(TS_COL).sort_index()
resampler = train_clean.resample(f'{SEQ_WINDOW_S}s', origin='epoch')

seq_len_sub = SEQ_WINDOW_S // SUBSAMPLE
n_channels = len(UNSUP_CHANNELS)

# First pass: count valid windows
valid_windows = []
for window_start, group in resampler:
    if len(group) < int(SEQ_WINDOW_S * 0.8):
        continue
    has_failure = group[FAIL_COL].astype(bool).any() if FAIL_COL in group.columns else False
    valid_windows.append((window_start, int(has_failure)))

print(f'  Valid windows: {len(valid_windows)}')

# Pre-allocate arrays (subsampled)
train_sequences = np.zeros((len(valid_windows), seq_len_sub, n_channels), dtype=np.float32)
train_seq_labels = np.zeros(len(valid_windows), dtype=np.int8)
train_seq_times = []

# Second pass: fill arrays
idx = 0
for window_start, group in resampler:
    if len(group) < int(SEQ_WINDOW_S * 0.8):
        continue
    
    vals = group[UNSUP_CHANNELS].values  # (n_obs, n_channels)
    
    # Subsample first, then interpolate to fixed length
    if len(vals) >= SEQ_WINDOW_S:
        subsampled = vals[::SUBSAMPLE][:seq_len_sub]
    else:
        # Interpolate to full length, then subsample
        x_old = np.linspace(0, 1, len(vals))
        x_new = np.linspace(0, 1, SEQ_WINDOW_S)
        full = np.zeros((SEQ_WINDOW_S, n_channels), dtype=np.float32)
        for ch in range(n_channels):
            full[:, ch] = np.interp(x_new, x_old, vals[:, ch])
        subsampled = full[::SUBSAMPLE][:seq_len_sub]
    
    # Pad if needed
    if len(subsampled) < seq_len_sub:
        pad = np.zeros((seq_len_sub - len(subsampled), n_channels), dtype=np.float32)
        subsampled = np.vstack([subsampled, pad])
    
    train_sequences[idx] = np.nan_to_num(subsampled, 0.0)
    train_seq_labels[idx] = valid_windows[idx][1]
    train_seq_times.append(window_start + pd.Timedelta(seconds=SEQ_WINDOW_S))
    idx += 1

print(f'Train sequences: {train_sequences.shape} | Failure: {train_seq_labels.sum()}')
# Memory: ~9872 * 350 * 95 * 4 bytes = ~1.2 GB (manageable)

del train_clean; gc.collect()

# Split into train-normal (80%) and val-normal (20%)
normal_mask = train_seq_labels == 0
normal_seqs = train_sequences[normal_mask]
n_train_normal = int(len(normal_seqs) * 0.8)
unsup_train = normal_seqs[:n_train_normal]
unsup_val   = normal_seqs[n_train_normal:]
print(f'Unsupervised train: {unsup_train.shape} | val: {unsup_val.shape}')

# Save
os.makedirs(os.path.join(OUT_DIR, 'unsupervised'), exist_ok=True)
np.save(os.path.join(OUT_DIR, 'unsupervised', 'train.npy'), unsup_train)
np.save(os.path.join(OUT_DIR, 'unsupervised', 'val.npy'), unsup_val)

with open(os.path.join(OUT_DIR, 'unsupervised', 'metadata.json'), 'w') as f:
    json.dump({'channels': UNSUP_CHANNELS, 'seq_len_s': SEQ_WINDOW_S,
               'subsample': SUBSAMPLE, 'effective_seq_len': seq_len_sub,
               'channel_stats': channel_stats}, f, indent=2)

del train_sequences, normal_seqs; gc.collect()
print('Train sequences saved.')

In [ ]:
# ── Create test sequences ──
print('Loading test data for unsupervised sequences...')
test_files = sorted(glob.glob(TEST_GLOB, recursive=True))

chunks = []
for f in test_files:
    df = pd.read_parquet(f)
    avail = [c for c in unsup_cols if c in df.columns]
    chunks.append(df[avail])
    del df

test_raw = pd.concat(chunks, ignore_index=True)
del chunks; gc.collect()

test_raw[TS_COL] = pd.to_datetime(test_raw[TS_COL])
test_raw.sort_values(TS_COL, inplace=True)
test_raw.reset_index(drop=True, inplace=True)

# Filter maintenance
test_clean = test_raw[~test_raw[MAINT_COL].astype(bool)].copy()
del test_raw; gc.collect()

# z-score standardize with TRAIN-NORMAL stats
print('Standardizing test data...')
for c in UNSUP_CHANNELS:
    test_clean[c] = (test_clean[c] - channel_stats[c]['mean']) / channel_stats[c]['std']

# Create sequences (subsampled)
print(f'Creating test sequences (subsample={SUBSAMPLE})...')
test_clean = test_clean.set_index(TS_COL).sort_index()
resampler = test_clean.resample(f'{SEQ_WINDOW_S}s', origin='epoch')

# First pass: count
valid_test = []
for window_start, group in resampler:
    if len(group) < int(SEQ_WINDOW_S * 0.8):
        continue
    has_failure = group[FAIL_COL].astype(bool).any() if FAIL_COL in group.columns else False
    valid_test.append((window_start, int(has_failure)))

print(f'  Valid windows: {len(valid_test)}')

# Pre-allocate
test_sequences = np.zeros((len(valid_test), seq_len_sub, n_channels), dtype=np.float32)
test_seq_labels = np.zeros(len(valid_test), dtype=np.int8)
test_seq_times_list = []

idx = 0
for window_start, group in resampler:
    if len(group) < int(SEQ_WINDOW_S * 0.8):
        continue
    
    vals = group[UNSUP_CHANNELS].values
    
    if len(vals) >= SEQ_WINDOW_S:
        subsampled = vals[::SUBSAMPLE][:seq_len_sub]
    else:
        x_old = np.linspace(0, 1, len(vals))
        x_new = np.linspace(0, 1, SEQ_WINDOW_S)
        full = np.zeros((SEQ_WINDOW_S, n_channels), dtype=np.float32)
        for ch in range(n_channels):
            full[:, ch] = np.interp(x_new, x_old, vals[:, ch])
        subsampled = full[::SUBSAMPLE][:seq_len_sub]
    
    if len(subsampled) < seq_len_sub:
        pad = np.zeros((seq_len_sub - len(subsampled), n_channels), dtype=np.float32)
        subsampled = np.vstack([subsampled, pad])
    
    test_sequences[idx] = np.nan_to_num(subsampled, 0.0)
    test_seq_labels[idx] = valid_test[idx][1]
    test_seq_times_list.append(window_start + pd.Timedelta(seconds=SEQ_WINDOW_S))
    idx += 1

test_seq_times_arr = np.array(test_seq_times_list)
print(f'Test sequences: {test_sequences.shape} | Failure: {test_seq_labels.sum()}')

np.save(os.path.join(OUT_DIR, 'unsupervised', 'test.npy'), test_sequences)
np.save(os.path.join(OUT_DIR, 'unsupervised', 'test_labels.npy'), test_seq_labels)
np.save(os.path.join(OUT_DIR, 'unsupervised', 'test_times.npy'), test_seq_times_arr)

del test_clean; gc.collect()
print('Test sequences saved.')

## 4. LSTM-AE Training

Simple LSTM autoencoder trained on normal-only sequences.
Anomaly score = mean reconstruction error per sequence.

In [ ]:
# ── LSTM-AE Model Definition ──
HIDDEN_DIM = 64
N_EPOCHS = 10  # reduced from 20 for CPU feasibility
BATCH_SIZE = 128  # larger batches = fewer forward passes
LR = 1e-3
N_CHANNELS = len(UNSUP_CHANNELS)


class LSTMAutoencoder(nn.Module):
    def __init__(self, n_channels, hidden_dim):
        super().__init__()
        self.encoder = nn.LSTM(n_channels, hidden_dim, batch_first=True)
        self.decoder = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)
        self.output_layer = nn.Linear(hidden_dim, n_channels)
    
    def forward(self, x):
        _, (h, c) = self.encoder(x)
        seq_len = x.size(1)
        decoder_input = h.squeeze(0).unsqueeze(1).repeat(1, seq_len, 1)
        decoder_out, _ = self.decoder(decoder_input, (h, c))
        reconstruction = self.output_layer(decoder_out)
        return reconstruction


print(f'LSTM-AE config: hidden={HIDDEN_DIM}, epochs={N_EPOCHS}, batch={BATCH_SIZE}, channels={N_CHANNELS}')
print(f'Sequence length: {SEQ_WINDOW_S // SUBSAMPLE}')

In [ ]:
# ── Train LSTM-AE ──
# Data from cell-10 is already subsampled to (N, 350, 95) — no further subsampling needed
unsup_train = np.load(os.path.join(OUT_DIR, 'unsupervised', 'train.npy'))
unsup_val   = np.load(os.path.join(OUT_DIR, 'unsupervised', 'val.npy'))

print(f'Training data: {unsup_train.shape}')
print(f'Validation data: {unsup_val.shape}')

# Replace NaN with 0
train_data = np.nan_to_num(unsup_train, 0.0)
val_data   = np.nan_to_num(unsup_val, 0.0)

# Create DataLoaders
train_tensor = torch.FloatTensor(train_data)
val_tensor   = torch.FloatTensor(val_data)

train_loader = DataLoader(TensorDataset(train_tensor, train_tensor), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(val_tensor, val_tensor), batch_size=BATCH_SIZE)

# Initialize model
device = torch.device('cpu')
model_ae = LSTMAutoencoder(N_CHANNELS, HIDDEN_DIM).to(device)
optimizer = torch.optim.Adam(model_ae.parameters(), lr=LR)
criterion = nn.MSELoss()

# Training loop
print('\nTraining LSTM-AE...')
train_losses = []
val_losses = []

for epoch in range(N_EPOCHS):
    model_ae.train()
    epoch_loss = 0
    n_batches = 0
    for x_batch, _ in train_loader:
        x_batch = x_batch.to(device)
        recon = model_ae(x_batch)
        loss = criterion(recon, x_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1
    train_loss = epoch_loss / n_batches
    train_losses.append(train_loss)
    
    model_ae.eval()
    val_loss = 0
    n_val = 0
    with torch.no_grad():
        for x_batch, _ in val_loader:
            x_batch = x_batch.to(device)
            recon = model_ae(x_batch)
            val_loss += criterion(recon, x_batch).item()
            n_val += 1
    val_loss = val_loss / n_val
    val_losses.append(val_loss)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'  Epoch {epoch+1:2d}/{N_EPOCHS}: train_loss={train_loss:.6f} val_loss={val_loss:.6f}')

# Save model
torch.save(model_ae.state_dict(), os.path.join(OUT_DIR, 'unsupervised', 'lstm_ae_model.pt'))
print('\nLSTM-AE training complete. Model saved.')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_losses, label='Train loss')
ax.plot(val_losses, label='Val loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('LSTM-AE Training Curve')
ax.legend()
plt.tight_layout()
plt.show()

del unsup_train, unsup_val, train_data, val_data, train_tensor, val_tensor
gc.collect()

In [ ]:
# ── Compute LSTM-AE anomaly scores on test sequences ──
# Data is already subsampled to (N, 350, 95) — no further subsampling needed
test_sequences = np.load(os.path.join(OUT_DIR, 'unsupervised', 'test.npy'))
test_seq_labels = np.load(os.path.join(OUT_DIR, 'unsupervised', 'test_labels.npy'))
test_seq_times_arr = np.load(os.path.join(OUT_DIR, 'unsupervised', 'test_times.npy'), allow_pickle=True)

test_clean_data = np.nan_to_num(test_sequences, 0.0)
test_tensor = torch.FloatTensor(test_clean_data)
test_loader = DataLoader(TensorDataset(test_tensor, test_tensor), batch_size=BATCH_SIZE)

# Load model
model_ae = LSTMAutoencoder(N_CHANNELS, HIDDEN_DIM)
model_ae.load_state_dict(torch.load(os.path.join(OUT_DIR, 'unsupervised', 'lstm_ae_model.pt'), weights_only=True))
model_ae.eval()

# Compute reconstruction errors
print('Computing reconstruction errors on test sequences...')
all_errors = []
with torch.no_grad():
    for x_batch, _ in test_loader:
        recon = model_ae(x_batch)
        errors = ((x_batch - recon) ** 2).mean(dim=(1, 2)).numpy()
        all_errors.append(errors)

anomaly_scores = np.concatenate(all_errors)
print(f'Anomaly scores: {anomaly_scores.shape}')
print(f'  Normal  mean: {anomaly_scores[test_seq_labels == 0].mean():.6f}')
print(f'  Failure mean: {anomaly_scores[test_seq_labels == 1].mean():.6f}')

# Save LSTM-AE test results
lstmae_results = pd.DataFrame({
    'window_end': test_seq_times_arr,
    'anomaly_score_lstmae': anomaly_scores,
    'true_failure_label': test_seq_labels,
})
lstmae_results.to_csv(os.path.join(OUT_DIR, 'unsupervised', 'test_predictions_lstmae.csv'), index=False)
print(f'\nSaved LSTM-AE predictions: {len(lstmae_results)} sequences')

del test_sequences, test_clean_data, test_tensor; gc.collect()

## 5. Final Anomaly Dataset

Merge RF and LSTM-AE results into a unified anomaly timeline.

In [ ]:
# ── Load both prediction sets ──
rf_preds = pd.read_csv(os.path.join(OUT_DIR, 'supervised', 'test_predictions_rf.csv'))
rf_preds['window_end'] = pd.to_datetime(rf_preds['window_end'])
rf_preds['timestamp'] = pd.to_datetime(rf_preds['timestamp'])

lstmae_preds = pd.read_csv(os.path.join(OUT_DIR, 'unsupervised', 'test_predictions_lstmae.csv'))
lstmae_preds['window_end'] = pd.to_datetime(lstmae_preds['window_end'])

print(f'RF predictions: {len(rf_preds)} windows (Wfeat={WINDOW_S}s)')
print(f'LSTM-AE predictions: {len(lstmae_preds)} sequences (Wseq={SEQ_WINDOW_S}s)')

# Note: RF and LSTM-AE have different temporal resolutions
# RF: 350s windows, LSTM-AE: 1402s windows
# We keep them separate but save a summary

print('\n=== RF Test Summary ===')
print(f'  Total windows: {len(rf_preds)}')
print(f'  True failures: {rf_preds["true_failure_label"].sum()}')
print(f'  Predicted failures: {rf_preds["predicted_failure_label"].sum()}')
print(f'  Date range: {rf_preds["timestamp"].min()} to {rf_preds["timestamp"].max()}')

print('\n=== LSTM-AE Test Summary ===')
print(f'  Total sequences: {len(lstmae_preds)}')
print(f'  True failures: {lstmae_preds["true_failure_label"].sum()}')
print(f'  Mean anomaly score (normal):  {lstmae_preds.loc[lstmae_preds["true_failure_label"]==0, "anomaly_score_lstmae"].mean():.6f}')
print(f'  Mean anomaly score (failure): {lstmae_preds.loc[lstmae_preds["true_failure_label"]==1, "anomaly_score_lstmae"].mean():.6f}')

In [ ]:
# ── Timeline visualization ──
saved = joblib.load(os.path.join(OUT_DIR, 'supervised', 'rf_model.joblib'))
rf_threshold = saved['threshold']

fig, axes = plt.subplots(2, 1, figsize=(20, 8), sharex=True)

# Plot 1: RF probability of failure
ax = axes[0]
ax.scatter(rf_preds.loc[rf_preds['true_failure_label']==0, 'window_end'],
           rf_preds.loc[rf_preds['true_failure_label']==0, 'probability_of_failure'],
           s=1, alpha=0.3, c='blue', label='Normal')
ax.scatter(rf_preds.loc[rf_preds['true_failure_label']==1, 'window_end'],
           rf_preds.loc[rf_preds['true_failure_label']==1, 'probability_of_failure'],
           s=5, alpha=0.8, c='red', label='Failure')
ax.axhline(y=rf_threshold, color='green', linestyle='--', linewidth=1, label=f'Threshold={rf_threshold:.2f}')
ax.set_ylabel('P(failure)')
ax.set_title('RF Failure Probability — Full Test Period')
ax.legend(loc='upper right')

# Plot 2: LSTM-AE anomaly score
ax = axes[1]
ax.scatter(lstmae_preds.loc[lstmae_preds['true_failure_label']==0, 'window_end'],
           lstmae_preds.loc[lstmae_preds['true_failure_label']==0, 'anomaly_score_lstmae'],
           s=1, alpha=0.3, c='blue', label='Normal')
ax.scatter(lstmae_preds.loc[lstmae_preds['true_failure_label']==1, 'window_end'],
           lstmae_preds.loc[lstmae_preds['true_failure_label']==1, 'anomaly_score_lstmae'],
           s=5, alpha=0.8, c='red', label='Failure')
ax.set_ylabel('Reconstruction Error')
ax.set_title('LSTM-AE Anomaly Score — Full Test Period')
ax.set_xlabel('Time')
ax.legend(loc='upper right')

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
axes[-1].xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'anomaly_timeline.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Timeline plot saved to outputs/anomaly_timeline.png')

In [ ]:
# ── Save summary ──
# Compute RF test metrics from predictions
y_true_rf = rf_preds['true_failure_label'].astype(int)
y_pred_rf = rf_preds['predicted_failure_label'].astype(int)
f1_t = f1_score(y_true_rf, y_pred_rf, zero_division=0)
p_t = precision_score(y_true_rf, y_pred_rf, zero_division=0)
r_t = recall_score(y_true_rf, y_pred_rf, zero_division=0)
cm = confusion_matrix(y_true_rf, y_pred_rf)
tn, fp, fn, tp = cm.ravel()
far = fp / (fp + tn) if (fp + tn) > 0 else 0

summary = {
    'supervised': {
        'window_s': WINDOW_S,
        'horizon_s': HORIZON_S,
        'n_analog': len(ANALOG_SENSORS),
        'n_binary': len(BINARY_SENSORS),
        'n_features': len(ANALOG_SENSORS)*4 + len(BINARY_SENSORS)*2 + 2,
        'test_windows': int(len(rf_preds)),
        'test_true_failures': int(rf_preds['true_failure_label'].sum()),
        'test_predicted_failures': int(rf_preds['predicted_failure_label'].sum()),
        'rf_threshold': float(rf_threshold),
        'rf_test_f1': float(f1_t),
        'rf_test_precision': float(p_t),
        'rf_test_recall': float(r_t),
        'rf_test_far': float(far),
        'rf_best_params': saved['params'],
    },
    'unsupervised': {
        'seq_window_s': SEQ_WINDOW_S,
        'n_channels': N_CHANNELS,
        'subsample': SUBSAMPLE,
        'hidden_dim': HIDDEN_DIM,
        'n_epochs': N_EPOCHS,
        'test_sequences': int(len(lstmae_preds)),
        'test_failure_sequences': int(lstmae_preds['true_failure_label'].sum()),
    }
}

with open(os.path.join(OUT_DIR, 'anomaly_dataset_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print('Summary saved to outputs/anomaly_dataset_summary.json')
print('\n=== Done ===')
print('\nOutputs:')
print('  outputs/supervised/train.parquet           — supervised train features')
print('  outputs/supervised/test.parquet            — supervised test features')
print('  outputs/supervised/test_predictions_rf.csv — RF predictions on ALL test windows')
print('  outputs/supervised/rf_model.joblib         — trained RF model')
print('  outputs/unsupervised/train.npy             — normal-only train sequences')
print('  outputs/unsupervised/val.npy               — normal-only val sequences')
print('  outputs/unsupervised/test.npy              — all test sequences')
print('  outputs/unsupervised/test_predictions_lstmae.csv — LSTM-AE anomaly scores')
print('  outputs/unsupervised/lstm_ae_model.pt      — trained LSTM-AE model')
print('  outputs/anomaly_timeline.png               — timeline visualization')
print('  outputs/anomaly_dataset_summary.json       — summary statistics')